# 🎯 프로젝트 설계 종합 정리 노트 (7장 — 주제 선정 · 시스템 설계 · 준비)

> **생성형 AI 기반 음성 에이전트 개발 과정 · 7장 프로젝트 파트 리뷰**
> `Modules/` 폴더 실습 노트북 중 **7장 2종**(7-1 주제 선정 · 7-2 시스템 설계 및 준비)을
> 하나로 정리한 **복습·재사용용 노트**입니다. (6장 시스템 통합은 `07-system-integration.ipynb`에서 정리)

| 항목 | 내용 |
|---|---|
| 대상 노트북 | `Modules/` 7-1 · 7-2 |
| 노트의 목적 | ① **선행 지식** ② **함수/클래스** 정의·주석 ③ **실험 진행 방법** ④ **효율적 설계 아키텍처** |
| 실행 환경 | **macOS (Apple Silicon)** — 게이트·계약서·파생 산출물 전부 로컬 실행 |
| 관계 노트 | **system-integration** 계약·예산 · **LLM 1권** 사고부 · **TTS/Cloning** 계약 |

> ⚙️ **실행 안내** — 이 노트의 코드 셀은 **GPU·모델·키·네트워크 없이** 실행되는 4대 게이트·계약서 검증·
> 파생 산출물(스키마·대화계획·시나리오·스파이크·마일스톤·스텁)만 모았습니다.
> 실제 모델 로드·pytest 실행은 시그니처+가이드로 요약했습니다 (📄). 위에서 아래로 실행하세요.


## 📑 목차
| 장 | 내용 |
|---|---|
| **0** | 노트북 로드맵 (7장 2종 · macOS 실행 판정) |
| **1** | 실험에 필요한 선행 지식 (4대 게이트 · 계약서 · 파생 산출물) |
| **2** | 함수/클래스 정의 및 주석 (실행 코드 ✅ + 요약 📄) |
| **3** | 실험 진행 방법 (주제 선정 · 설계 파생 + macOS 가이드) |
| **4** | 효율적 설계를 위한 아키텍처 |


# 0. 노트북 로드맵 🗺️

## 0-1. 7장 2종 한눈에

| 노트북 | 주제 | **macOS 판정** | 코멘트 |
|---|---|---|---|
| **7-1** | 프로젝트 주제 선정 | ✅ 전부 실행 | 4대 게이트 · 계약서(Charter) · 주제 채점 |
| **7-2** | 시스템 설계 및 준비 | ✅ 전부 실행 | 스키마 파생 · 대화 계획 · 시나리오 · 스파이크 · 스텁 |

> **핵심 흐름**: 7-1(주제가 산수와 라이선스를 통과하는가) → 7-2(계약서를 정본으로 설계 산출물을 *파생*한다).
> **"검증 없이 서명 없다"** — 계약서는 서명한 날이 아니라 **검사하는 날마다** 유효하다.

## 0-2. 4대 게이트 — 탈락은 산수가 결정한다

```
Gate1 VRAM:  Σ 모델 vram_gb × 1.2(여유) + 1.5(CUDA) ≤ 16GB(T4)
Gate2 TTFA:  ASR + LLM 첫문장 + TTS 첫청크 ≤ 상호작용 예산 (1500ms 실시간)
Gate3 라이선스: 조합 = max(구성원 랭크) — '미확인'은 어떤 의도로도 통과 불가
Gate4 골든셋:  의도 5종 × 발화 3개 · snake_case · 8~80자 · 중복 금지
```


## 📖 0-A. 용어 사전 & 배경 지식 — 이 노트를 처음 읽는 사람을 위한 지도

> **이 노트를 처음 공부하는 방법**: ① 0-A 용어사전 훑기 → ② 1장 선행 지식 → ③ 2장 함수 실행하며
> "검증 통과 ✅" 눈으로 확인. 모르는 단어는 여기로 돌아오세요.

### A. 4대 게이트 — "주제는 산수가 심사한다"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| Gate1 VRAM | 모델 조합 메모리 합계 ≤ T4(16GB) | 탈락은 주장이 아니라 합계가 결정 |
| Gate2 TTFA | ASR+LLM+TTS 첫 결과 지연 ≤ 예산 | 실시간이면 1500ms 안 |
| Gate3 라이선스 | 조합 = max(구성원 랭크) | '미확인'은 어떤 의도로도 통과 불가 |
| Gate4 골든셋 | 의도 5종 × 발화 3개 규격 검사 | 평가 데이터 자체가 계약 |
| SAFETY_MARGIN | VRAM 여유 20% | 활성값·단편화 대비 |
| INTERACTION_BUDGETS | 상호작용별 TTFA 예산 | 실시간 1500 / 통지 3000 / 배치 없음 |

### B. 계약서 Charter — "범위는 non-goal이 지킨다"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| Charter (계약서) | 프로젝트의 12필드 정본 | 모든 파생 산출물의 출처(SSOT) |
| one_liner | 80자 이내 한 문장 정의 | "두 문장이면 주제가 둘" |
| success_criteria | 숫자+단위가 있는 성공 기준 | 측정 불가능한 기준은 없다 |
| non_goals | "안 하는 것" 3개 이상 | 범위는 하는 게 아니라 안 하는 게 지킨다 |
| risks | 리스크 3종 + 완화책 필수 | 완화책 없는 리스크는 걱정이지 관리가 아니다 |
| verification_log | 재검증 날짜 기록 | "검증 없이 서명 없다" |
| MEASURABLE | 숫자+단위 정규식 | 성공 기준 검사 도구 |

### C. 설계 파생 — "손 파생은 배신한다"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| SSOT (정본) | 단일 진실 공급원 = Charter | 사본은 함수가 파생 |
| check_schema_consistency | 정본과 사본(enum) 일치 검사 | 손으로 베낀 스키마는 빠뜨린다 |
| dialog_plan | 의도별 슬롯·확인·handoff 계획 | 대화 설계의 정본 |
| acceptance_scenarios | 골든셋 × 계획 → 기대 행동표 | 7-3 완료 정의 · 7-5 시연 대본 |
| mermaid | 대화 흐름을 텍스트로 그린 다이어그램 | 그림을 손으로 그리면 어긋난다 |
| spike | 리스크별 '2시간짜리 실험' | 숫자 판정 기준 필수 |
| milestone | 기한+판정이 있는 마일스톤 | pytest 판정 2개 이상 |
| stub | NotImplementedError 스텁 | 7-3의 작업 지시서 |

### D. 배경 지식 — 이 챕터가 왜 존재하는가
앞의 6장까지가 "부품을 만드는 법"이었다면, **7장은 "완성될 수 있는 프로젝트를 고르고 설계하는 법"**입니다.
1. **완성 가능성은 산수** — 모델 조합이 메모리·지연·라이선스를 통과하지 못하면 야심이 아니라 환상.
   "미완성 야심작과 완성된 수수한 작품 중 PoC 시연에서 이기는 쪽은 언제나 후자다."
2. **정본은 하나** — 계약서(Charter)를 SSOT로 두고, 스키마·시나리오·스텁은 **함수가 파생**한다.
   손으로 베낀 사본은 `inquiry` 하나를 빠뜨리듯 배신한다.
3. **증명은 종료코드** — 스파이크·마일스톤의 판정은 사람의 인상이 아니라 숫자(판정 기준)와
   pytest 종료코드로 정한다.


### 0-B. 2장 함수 지도 — 어떤 셀이 무슨 역할인지 미리 보기
| 셀 | 함수/클래스 | 역할 한 줄 | 핵심 개념 |
|---|---|---|---|
| 2.0 | `vram_check`·`ttfa_estimate`·`license_check`·`validate_golden_set` | 4대 게이트 | 산수·정책으로 심사 |
| 2.1 | `score_topics`·`validate_charter` | 주제 채점 + 계약서 검증 | 가중 합 · 12필드 |
| 2.2 | `check_schema_consistency`·`derive_project_schema`·`validate_dialog_plan` | SSOT 파생 | 손 파생 배신 차단 |
| 2.3 | `derive_acceptance_scenarios`·`mermaid_sequence`·`validate_spikes`·`validate_milestones` | 시나리오·계획 파생 | 숫자 판정 |
| 2.4 | `stub_source`·계약 스켈레톤 | 스텁 파생 | 7-3 작업 지시서 |


# 1. 실험에 필요한 선행 지식 🧠

## 1-1. 모델 카탈로그 (7-1) — 시점 기록, 재검증 의무

| 모델 | kind | vram_gb | first_ms | 라이선스 |
|---|---|---|---|---|
| faster-whisper-small(int8) | asr | 0.7 | 400 | 상업 자유 |
| faster-whisper-large-v3 | asr | 3.2 | 900 | 상업 자유 |
| EXAONE-4.0-1.2B(fp16) | llm | 2.6 | 600 | 조건부 상업 |
| Midm-2.0-Mini(fp16) | llm | 4.8 | 700 | 조건부 상업 |
| Bllossom-3B(fp16) | llm | 6.5 | 800 | 상업 자유 |
| Qwen3-8B(4bit) | llm | 6.0 | 1200 | 상업 자유 |
| Qwen3-8B(fp16) | llm | 16.5 | 1000 | 상업 자유 (T4 탈락 확정) |
| Gemini-API | llm | 0.0 | 700 | API 약관 (네트워크·쿼터) |
| Kokoro(82M) | tts | 0.5 | 300 | 상업 자유 |
| F5-TTS | tts | 1.5 | 900 | 비상업(NC) |
| Chatterbox-Multilingual | tts | 2.2 | 800 | 상업 자유 |
| Qwen2.5-Omni-3B | omni | 8.5 | 1100 | 상업 자유 (일체형) |

> **핵심**: `src` 필드가 어느 실습에서 측정했는지 기록 — "재검증 날짜"를 계약서가 강제한다.

## 1-2. 상호작용 예산 (Gate2)

| interaction | ttfa_ms | 의미 |
|---|---|---|
| realtime_inbound | 1500 | 고객이 침묵을 견디는 시간 (6-2 예산 그대로) |
| outbound_notify | 3000 | 발신 통지 — 상용구 캐시로 은폐 가능 |
| offline_batch | 없음 | 비실시간 — TTFA 무의미, **처리량**이 지표 |

## 1-3. 계약서 Charter (7-1) — 범위는 non-goal이 지킨다

필수 12필드: `project_name · one_liner · target_user · interaction_type · intents · golden_set ·
success_criteria · non_goals · model_stack · deployment_intent · risks · verification_log`

- **한 줄 정의**: 80자 이내·문장 1개 — "두 문장이 필요하면 아직 주제가 둘이다".
- **측정 가능성**: 성공 기준마다 숫자+단위(`MEASURABLE`) — "측정 불가능한 기준은 없다".
- **non_goals 3개+**: 범위는 "하는 것"이 아니라 "안 하는 것"이 지킨다.
- **리스크 3개+**: 완화책이 없는 리스크는 걱정이지 관리가 아니다.
- **재검증 날짜**: `model_cards_checked`·`license_checked` — "검증 없이 서명 없다".

## 1-4. 파생 산출물 (7-2) — 손 파생의 배신, 정본에서 파생한다

```
계약서(Charter) = SSOT(단일 진실 공급원)
  ├─ INTENT_SCHEMA 프로젝트판  (derive_project_schema)
  ├─ dialog_plan (대화 계획)   → mermaid_sequence + acceptance_scenarios
  ├─ spikes (2시간짜리 실험)   → 리스크별 판정 기준(숫자!)
  ├─ milestones (마일스톤)     → pytest 판정 2개 이상
  └─ project_engines.py 스텁   → 7-3 구현의 작업 지시서 (NotImplementedError)
```

**교훈 (7-2 셀 2-1)**: 손으로 베낀 스키마는 배신한다 (`inquiry` 누락) — `check_schema_consistency`로 정본과 사본의
어긋남을 먼저 잡는다. 스키마는 **손이 아니라 함수가 만든다**.

## 1-5. macOS 실행 가이드

| 항목 | 판정 | 설명 |
|---|---|---|
| 4대 게이트·계약서·파생 함수 (7-1/7-2) | ✅ 전부 실행 | 이 노트 2.0~2.4 |
| pytest 스위트 (7-2 셀 8) | ⚠️ pip 필요 | `pip install pytest` — 로컬 실행 |
| 모델 실측(VRAM·TTFA) | ⚠️ 모델 필요 | 카탈로그 수치는 T4 기준 — macOS는 **fp16 MPS로 재측정** |

> **macOS 관점**: 게이트 로직은 어디서든 동일. 다만 `Qwen3-8B(fp16)`은 T4에서 탈락하는 수치지만
> **M4 Pro 48GB에서는 fp16(≈16GB) 상주 가능** — 하드웨어가 바뀌면 게이트 재평가가 필요하다는 뜻.


# 2. 함수/클래스 정의 및 주석 🔧

> ✅ = 실행 코드 셀 (GPU·모델·키 없이 assert 자가점검) · 📄 = 요약만 (실물 로드·pytest는 3장 가이드)


In [ ]:
# ═══ 2.0 4대 게이트 — VRAM · TTFA · 라이선스 · 골든셋 (7-1) ✅ ═══
# ▶ 4대 게이트: VRAM(합계×1.2+1.5) · TTFA(단계 합) · 라이선스(최대 랭크) · 골든셋(규격).
#   각 게이트를 '정상/위반' 양쪽으로 검증 — 게이트가 실제로 막는지 확인한다.
import json, re, math

T4_VRAM_GB, SAFETY_MARGIN, CUDA_OVERHEAD_GB = 16.0, 1.20, 1.5
GOLDEN_MIN_INTENTS, GOLDEN_MIN_UTTS = 5, 3
LICENSE_RANK = {"상업 자유": 0, "조건부 상업": 1, "API 약관": 1,
                "비상업(NC)": 2, "연구 전용": 3, "미확인": 4}
RANK_NAME = {0: "상업 자유", 1: "조건부/API", 2: "비상업(NC)", 3: "연구 전용", 4: "미확인"}
MEASURABLE = re.compile(r"\d+(\.\d+)?\s*(ms|초|s|%|점|건|회|자|일|분|턴)")
DATE_RE = re.compile(r"20\d\d-\d\d-\d\d")

MODEL_CATALOG = {
    "faster-whisper-small(int8)": {"kind": "asr", "vram_gb": 0.7, "first_ms": 400, "license": "상업 자유"},
    "faster-whisper-large-v3":    {"kind": "asr", "vram_gb": 3.2, "first_ms": 900, "license": "상업 자유"},
    "EXAONE-4.0-1.2B(fp16)":      {"kind": "llm", "vram_gb": 2.6, "first_ms": 600, "license": "조건부 상업"},
    "Midm-2.0-Mini(fp16)":        {"kind": "llm", "vram_gb": 4.8, "first_ms": 700, "license": "조건부 상업"},
    "Bllossom-3B(fp16)":          {"kind": "llm", "vram_gb": 6.5, "first_ms": 800, "license": "상업 자유"},
    "Qwen3-8B(4bit)":             {"kind": "llm", "vram_gb": 6.0, "first_ms": 1200, "license": "상업 자유"},
    "Qwen3-8B(fp16)":             {"kind": "llm", "vram_gb": 16.5, "first_ms": 1000, "license": "상업 자유"},
    "Gemini-API":                 {"kind": "llm", "vram_gb": 0.0, "first_ms": 700, "license": "API 약관"},
    "Kokoro(82M)":                {"kind": "tts", "vram_gb": 0.5, "first_ms": 300, "license": "상업 자유"},
    "F5-TTS":                     {"kind": "tts", "vram_gb": 1.5, "first_ms": 900, "license": "비상업(NC)"},
    "Chatterbox-Multilingual":    {"kind": "tts", "vram_gb": 2.2, "first_ms": 800, "license": "상업 자유"},
    "Qwen2.5-Omni-3B":            {"kind": "omni", "vram_gb": 8.5, "first_ms": 1100, "license": "상업 자유"},
}
INTERACTION_BUDGETS = {
    "realtime_inbound": 1500,
    "outbound_notify":  3000,
    "offline_batch":    None,
}

def vram_check(stack, verbose=True):
    # 모델 조합 VRAM 합계 vs T4 — 탈락은 산수가 결정한다 (다른 평가보다 먼저)
    rows, total = [], 0.0
    for name in stack:
        m = MODEL_CATALOG[name]
        rows.append((name, m["vram_gb"])); total += m["vram_gb"]
    need = total * SAFETY_MARGIN + CUDA_OVERHEAD_GB
    ok = need <= T4_VRAM_GB
    if verbose:
        for name, gb in rows:
            print(f"  {name:<30}{gb:>6.1f} GB")
        print(f"  합계 {total:>5.1f} ×1.2 + 1.5 = {need:>5.1f} vs T4 {T4_VRAM_GB} → {'통과' if ok else '탈락'}")
    return ok, round(need, 1)

def ttfa_estimate(stack, interaction, verbose=True):
    # 근사 TTFA = ASR + LLM 첫문장 + TTS 첫청크. 자릿수 판정용이지 정밀 예측이 아니다
    parts = {MODEL_CATALOG[n]["kind"]: MODEL_CATALOG[n]["first_ms"] for n in stack}
    if "omni" in parts:
        est = parts["omni"]                      # 일체형은 단일 경로
    else:
        est = parts.get("asr", 0) + parts.get("llm", 0) + parts.get("tts", 0)
    budget = INTERACTION_BUDGETS[interaction]
    ok = True if budget is None else est <= budget
    if verbose:
        b = "제한 없음" if budget is None else str(budget) + "ms"
        print(f"  추정 TTFA {est}ms vs 예산 {b} → {'통과' if ok else '탈락'}")
    return ok, est

def license_check(stack, deployment_intent, verbose=True):
    # 조합 라이선스 = max(구성원 랭크). '미확인'은 어떤 의도로도 통과 불가
    worst = max(LICENSE_RANK[MODEL_CATALOG[n]["license"]] for n in stack)
    if worst >= 4:
        ok = False
    elif deployment_intent == "상업":
        ok = worst <= 1
    else:                                       # 수업 PoC(비상업)
        ok = worst <= 2
    if verbose:
        print(f"  조합 판정: {RANK_NAME[worst]} / 배포 의도: {deployment_intent} → {'통과' if ok else '탈락'}")
    return ok, RANK_NAME[worst]

def validate_golden_set(gs):
    # 골든셋 게이트 — 위반은 그 자리에서 크게 실패
    assert isinstance(gs, dict) and len(gs) >= GOLDEN_MIN_INTENTS,         "골든셋 위반 — 의도 " + str(GOLDEN_MIN_INTENTS) + "종 미만"
    seen = set()
    for intent, utts in gs.items():
        assert re.fullmatch(r"[a-z_]+", intent), "골든셋 위반 — intent는 snake_case 영문: " + intent
        assert len(utts) >= GOLDEN_MIN_UTTS, f"골든셋 위반 — {intent} 발화 부족"
        for u in utts:
            assert 8 <= len(u) <= 80, f"골든셋 위반 — 발화 길이 8~80자: '{u}'"
            assert u not in seen, "골든셋 위반 — 중복 발화: '" + u + "'"
            seen.add(u)
    return True

# ── 게이트 양방향 검증 ──
stack_ok = ["faster-whisper-small(int8)", "EXAONE-4.0-1.2B(fp16)", "Kokoro(82M)"]
stack_bad_vram = ["faster-whisper-large-v3", "Qwen3-8B(fp16)", "Chatterbox-Multilingual"]  # fp16 → 탈락
ok1, need1 = vram_check(stack_ok, verbose=False)
ok1b, _ = vram_check(stack_bad_vram, verbose=False)
assert ok1 and not ok1b, f"Gate1 양방향: {need1}GB"

ok2, est2 = ttfa_estimate(stack_ok, "realtime_inbound", verbose=False)
ok2b, _ = ttfa_estimate(stack_bad_vram, "realtime_inbound", verbose=False)
assert ok2 and not ok2b

ok3, v3 = license_check(stack_ok, "수업 PoC(비상업)", verbose=False)
ok3b, _ = license_check(["F5-TTS"], "상업", verbose=False)
assert ok3 and not ok3b, f"Gate3: {v3}"

good_gs = {"reserve": ["다음 주 화요일 오전에 내과 예약하고 싶어요.",
                       "이번 주 금요일 오후 아무 때나 예약돼요.",
                       "아이 예방접종 예약하려고 전화드렸어요."],
           "change": ["내일 예약을 다음 주로 미루고 싶은데요.",
                      "오후 세 시 예약을 오전으로 바꿀 수 있을까요.",
                      "예약 날짜를 하루만 당길 수 있나요."],
           "cancel": ["갑자기 일이 생겨서 예약을 취소해야 할 것 같아요.",
                      "모레 잡아 둔 진료 예약 취소 부탁드립니다.",
                      "예약 취소하고 싶은데 위약금이 있나요."],
           "inquiry": ["토요일에도 진료하시나요.",
                       "주차장이 있는지 궁금해서요.",
                       "건강검진 전에 금식해야 하나요."],
           "handoff": ["며칠 전부터 배가 아픈데 어느 과로 가야 할까요.",
                       "복용 중인 약이 있는데 진료 전에 상담하고 싶어요.",
                       "검사 결과에 대해 물어볼 게 있어요."]}
validate_golden_set(good_gs)
try:
    validate_golden_set({"reserve": ["너무 짧다"]})   # 의도 부족 + 발화 길이 위반
    raise AssertionError("골든셋 위반 미포착")
except AssertionError:
    pass
print(f"4대 게이트 검증 통과 ✅ — VRAM {need1}GB / TTFA {est2}ms / 라이선스 {v3} / 골든셋 5×3")


In [ ]:
# ═══ 2.1 계약서 Charter — validate_charter · 주제 채점 (7-1) ✅ ═══
# ▶ validate_charter: 12필드 + 골든셋 + 측정가능 + non_goals + 4대 게이트 + 리스크 완화책.
#   score_topics: 가중 합 정렬 — '점수는 토론의 결과 기록이지 근거가 아니다'.
CRITERIA = {"고객 가치": 0.30, "실현 가능성(게이트)": 0.30,
            "데모 임팩트": 0.20, "차별성": 0.10, "데이터 확보": 0.10}
CHARTER_REQUIRED = ("project_name", "one_liner", "target_user", "interaction_type",
                    "intents", "golden_set", "success_criteria", "non_goals",
                    "model_stack", "deployment_intent", "risks", "verification_log")

def score_topics(scores: dict):
    # {주제: {기준: 1~5}} → 가중 합 정렬. 점수는 팀 토론의 '결과 기록'이지 근거가 아니다
    assert abs(sum(CRITERIA.values()) - 1.0) < 1e-9
    rows = []
    for topic, s in scores.items():
        missing = [c for c in CRITERIA if c not in s]
        assert not missing, "채점 누락: " + topic + " ← " + str(missing)
        total = sum(CRITERIA[c] * s[c] for c in CRITERIA)
        rows.append((round(total, 2), topic))
    return sorted(rows, reverse=True)

def validate_charter(c, verbose=True):
    missing = [k for k in CHARTER_REQUIRED if k not in c]
    assert not missing, "계약서 위반 — 누락 필드: " + str(missing)
    assert len(c["one_liner"]) <= 80 and c["one_liner"].count(".") <= 1,         "계약서 위반 — 한 줄 정의는 80자 이내 한 문장 (두 문장이면 아직 주제가 둘이다)"
    assert c["interaction_type"] in INTERACTION_BUDGETS, "계약서 위반 — interaction_type"
    assert len(c["intents"]) >= 5, "계약서 위반 — 핵심 의도 5종 미만"
    assert set(c["golden_set"].keys()) == set(c["intents"].keys()), "계약서 위반 — 골든셋/의도 불일치"
    validate_golden_set(c["golden_set"])
    assert len(c["success_criteria"]) >= 3, "계약서 위반 — 성공 기준 3개 미만"
    for sc in c["success_criteria"]:
        assert MEASURABLE.search(sc), "계약서 위반 — 측정 불가능한 성공 기준: '" + sc + "'"
    assert len(c["non_goals"]) >= 3, "계약서 위반 — non_goals 3개 미만 (범위는 non-goal이 지킨다)"
    stack = list(c["model_stack"].values())
    ok_v, need = vram_check(stack, verbose=False)
    assert ok_v, f"계약서 위반 — Gate1 VRAM 탈락 ({need}GB)"
    ok_t, est = ttfa_estimate(stack, c["interaction_type"], verbose=False)
    assert ok_t, f"계약서 위반 — Gate2 지연 탈락 ({est}ms)"
    ok_l, verdict = license_check(stack, c["deployment_intent"], verbose=False)
    assert ok_l, "계약서 위반 — Gate3 라이선스 탈락 (" + verdict + ")"
    assert len(c["risks"]) >= 3 and all(r.get("mitigation") for r in c["risks"]),         "계약서 위반 — 리스크 3종 + 완화책 필수 (완화책 없는 리스크는 걱정이지 관리가 아니다)"
    for key in ("model_cards_checked", "license_checked"):
        assert DATE_RE.fullmatch(c["verification_log"].get(key, "")),             "계약서 위반 — 재검증 날짜 누락: " + key + " (검증 없이 서명 없다)"
    if verbose:
        print(f"✅ 계약서 통과 — VRAM {need}GB / TTFA≈{est}ms / 라이선스 {verdict}")
    return True

# ── 예시 계약서 (메디콜 — 병원 예약 음성 접수) ──
EXAMPLE_CHARTER = {
    "project_name": "메디콜 — 병원 예약 음성 접수 에이전트",
    "one_liner": "환자가 전화로 진료 예약·변경·취소를 상담원 없이 90초 안에 끝내는 음성 에이전트.",
    "target_user": "중소 병원의 전화 예약 환자 (주 이용층 40~70대)",
    "interaction_type": "realtime_inbound",
    "deployment_intent": "수업 PoC(비상업)",
    "intents": {
        "reserve": "신규 진료 예약", "change": "기존 예약 일시 변경",
        "cancel": "예약 취소", "inquiry": "진료 시간·위치·준비물 문의",
        "handoff": "복잡 상담(증상 상담 등) — 사람 연결",
    },
    "golden_set": good_gs,
    "success_criteria": [
        "골든셋 15발화 라운드트립 의도 정확도 90% 이상",
        "실시간 시나리오 TTFA p95 1500ms 이하 (6-5 E2E 테스트로 판정)",
        "handoff 의도 3발화는 100% 사람 연결로 분기 (오분기 0건)",
        "예약 완료까지 평균 턴 수 6턴 이하",
    ],
    "non_goals": [
        "증상 기반 의료 상담·진단 (즉시 handoff — 의료법 리스크)",
        "실제 병원 EMR 연동 (PoC는 모의 예약 DB)",
        "다국어 지원 (한국어 단일)",
        "본인 인증·결제 (모의 확인 절차로 대체)",
    ],
    "model_stack": {"asr": "faster-whisper-small(int8)",
                    "llm": "EXAONE-4.0-1.2B(fp16)",
                    "tts": "Kokoro(82M)"},
    "risks": [
        {"risk": "고령 화자 발화의 ASR 정확도 저하",
         "mitigation": "reprompt 게이트(6-2) + 골든셋에 고령층 화법 발화 추가"},
        {"risk": "날짜·시간 슬롯 파싱 오류('다음 주 화요일')",
         "mitigation": "슬롯 정규화 유닛 테스트 선작성(7-3) + 확인 되묻기 턴 필수화"},
        {"risk": "EXAONE 조건부 라이선스의 상업 전환 제약",
         "mitigation": "어댑터 소켓 유지 — 상업 전환 시 Bllossom-3B로 교체"},
    ],
    "verification_log": {"model_cards_checked": "2026-08-03", "license_checked": "2026-08-03"},
}
validate_charter(EXAMPLE_CHARTER, verbose=True)

# ── 주제 채점 ──
scores = {
    "메디콜(병원 예약)": {"고객 가치": 5, "실현 가능성(게이트)": 5, "데모 임팩트": 4, "차별성": 3, "데이터 확보": 5},
    "실시간 동시통역 상담": {"고객 가치": 5, "실현 가능성(게이트)": 2, "데모 임팩트": 5, "차별성": 5, "데이터 확보": 3},
    "상담 녹취 이중자막": {"고객 가치": 3, "실현 가능성(게이트)": 5, "데모 임팩트": 3, "차별성": 3, "데이터 확보": 4},
}
ranked = score_topics(scores)
assert ranked[0][1] == "메디콜(병원 예약)"       # 가중 합 최고
print(f"계약서·채점 검증 통과 ✅ — 1위 {ranked[0][1]} {ranked[0][0]:.2f} / 2위 {ranked[1][1]} {ranked[1][0]:.2f}")


In [ ]:
# ▶ 데모 — '주제가 게이트를 통과하는지' 눈으로 확인 (초보자용)
# 모델 조합이 VRAM·TTFA·라이선스를 통과하는지, 실패하면 어느 게이트인지 보인다.

print("① 통과 조합 (faster-whisper-small + EXAONE-1.2B + Kokoro):")
ok_v, need = vram_check(["faster-whisper-small(int8)", "EXAONE-4.0-1.2B(fp16)", "Kokoro(82M)"], verbose=False)
ok_t, est = ttfa_estimate(["faster-whisper-small(int8)", "EXAONE-4.0-1.2B(fp16)", "Kokoro(82M)"],
                          "realtime_inbound", verbose=False)
ok_l, v = license_check(["faster-whisper-small(int8)", "EXAONE-4.0-1.2B(fp16)", "Kokoro(82M)"],
                        "수업 PoC(비상업)", verbose=False)
print(f"   VRAM {need}GB={'OK' if ok_v else 'X'} | TTFA {est}ms={'OK' if ok_t else 'X'} | 라이선스 {v}={'OK' if ok_l else 'X'}")

print("② 탈락 조합 (fp16 8B — VRAM 초과):")
need2 = vram_check(["faster-whisper-large-v3", "Qwen3-8B(fp16)", "Chatterbox-Multilingual"], verbose=False)
print(f"   VRAM 요구 {need2[1]}GB > 16GB → {'통과' if need2[0] else '탈락'} (fp16은 T4에서 못 돈다)")
assert ok_v and ok_t and ok_l
assert not need2[0]
print("데모 통과 ✅ — '탈락은 산수가 결정한다': 주장이 아니라 합계·추정·랭크로")


#### 📄 2.1 보충 — 주제 탈락 워크스루 (7-1)
"실시간 동시통역 상담" 스택(`large-v3 + Qwen3-8B(4bit) + Chatterbox`)은 VRAM은 통과하나 TTFA에서 탈락.
→ **"탈락과 용도는 다르다"**: 실시간을 버리고 `offline_batch`(상담 녹취 이중자막)로 용도를 바꾸면 통과.
> 미완성 야심작과 완성된 수수한 작품 중 PoC 시연(7-5)에서 이기는 쪽은 언제나 후자다.


In [ ]:
# ═══ 2.2 스키마 파생 + 대화 계획 — 손 파생의 배신 (7-2) ✅ ═══
# ▶ SSOT 파생: 스키마는 함수가 만들고, 손으로 베낀 사본(STALE)은 정합성 검사가 잡는다.
#   dialog_plan: handoff 문이 없으면 설계 결함 — '사람으로 도망갈 문'이 있어야 한다.
def check_schema_consistency(enum, charter):
    # 정본(계약서)과 사본(스키마 enum)의 어긋남을 잡는다
    a, b = set(enum), set(charter["intents"])
    assert a == b, ("스키마-계약서 불일치 — 스키마에 없음: " + str(sorted(b - a))
                    + " / 계약서에 없음: " + str(sorted(a - b)))
    return True

def derive_project_schema(charter):
    # 정본(계약서) → INTENT_SCHEMA 프로젝트판. 손이 아니라 이 함수만이 스키마를 만든다
    return {
        "type": "object",
        "properties": {
            "intent": {"type": "string", "enum": sorted(charter["intents"])},
            "slots": {"type": "object"},
            "reply": {"type": "string", "minLength": 1},
            "handoff_to_human": {"type": "boolean"},
        },
        "required": ["intent", "slots", "reply", "handoff_to_human"],
        "additionalProperties": False,
    }

def validate_dialog_plan(plan, charter):
    # 대화 계획 — 사람으로 도망갈 문(handoff)이 없으면 설계 결함
    a, b = set(plan), set(charter["intents"])
    assert a == b, "대화 계획 위반 — 의도 불일치: " + str(sorted(b - a)) + str(sorted(a - b))
    assert any(p.get("handoff") for p in plan.values()), "대화 계획 위반 — handoff 문이 없다"
    for intent, p in plan.items():
        for s in p["slots"]:
            assert s["question"].endswith("?") or s["question"].endswith("주세요."),                 "대화 계획 위반 — 슬롯 질문 형식: " + intent + "." + s["name"]
        if p["risky"]:
            assert p["confirm"] and p["confirm_line"],                 "대화 계획 위반 — 위험 의도 '" + intent + "'에 확인 턴 없음"
    return True

SCHEMA_PROJECT = derive_project_schema(EXAMPLE_CHARTER)
assert set(SCHEMA_PROJECT["properties"]["intent"]["enum"]) == set(EXAMPLE_CHARTER["intents"])

# 손으로 베낀 사본은 배신한다 — inquiry 누락 → 정합성 검사가 잡는다
STALE_SCHEMA_ENUM = ["reserve", "change", "cancel", "handoff"]
try:
    check_schema_consistency(STALE_SCHEMA_ENUM, EXAMPLE_CHARTER)
    raise AssertionError("손 파생 배신 미포착")
except AssertionError:
    pass

DIALOG_PLAN = {
    "reserve": {"slots": [{"name": "department", "question": "어느 진료과로 예약해 드릴까요?"},
                          {"name": "datetime", "question": "원하시는 날짜와 시간을 말씀해 주세요."}],
                "risky": True, "confirm": True,
                "confirm_line": "{datetime}에 {department} 예약을 확정할까요?"},
    "change":  {"slots": [{"name": "which", "question": "어느 예약을 변경하시겠어요?"},
                          {"name": "datetime", "question": "새로 원하시는 일시를 말씀해 주세요."}],
                "risky": True, "confirm": True,
                "confirm_line": "{which} 예약을 {datetime}로 변경할까요?"},
    "cancel":  {"slots": [{"name": "which", "question": "어느 예약을 취소하시겠어요?"}],
                "risky": True, "confirm": True,
                "confirm_line": "{which} 예약을 취소할까요? 취소 후에는 되돌릴 수 없습니다."},
    "inquiry": {"slots": [], "risky": False, "confirm": False, "confirm_line": ""},
    "handoff": {"slots": [], "risky": False, "confirm": False, "confirm_line": "", "handoff": True},
}
validate_dialog_plan(DIALOG_PLAN, EXAMPLE_CHARTER)
print("스키마 파생·대화 계획 검증 통과 ✅ — SSOT 파생 / 손 파생 배신 포착 / handoff 문 필수")


In [ ]:
# ═══ 2.3 수락 시나리오 + mermaid + 스파이크 + 마일스톤 (7-2) ✅ ═══
# ▶ 시나리오 = 골든셋 × 계획 → 기대 행동표 (완료 정의이자 시연 대본).
#   스파이크 판정엔 숫자(criterion), 마일스톤엔 pytest 판정 2개 이상이 필수.
def derive_acceptance_scenarios(charter, plan):
    # 골든셋 × 대화 계획 → 발화별 기대 행동표. 7-3 완료 정의이자 7-5 시연 대본의 씨앗
    rows = []
    for intent, utts in charter["golden_set"].items():
        p = plan[intent]
        if p.get("handoff"):
            expected = "즉시 사람 연결 안내 (handoff_to_human=true)"
        elif p["slots"]:
            expected = "첫 슬롯 질문: " + p["slots"][0]["question"]
        else:
            expected = "단답 응대 후 추가 문의 확인"
        final = "확인 턴 → 완료" if p["confirm"] else ("전화 전환" if p.get("handoff") else "응대 종료")
        for u in utts:
            rows.append({"utt": u, "intent": intent, "expected_first": expected, "expected_final": final})
    return rows

def mermaid_sequence(intent, plan):
    # dialog_plan → Mermaid 시퀀스 텍스트. 그림을 손으로 그리면 계획과 어긋난다
    p = plan[intent]
    L = ["sequenceDiagram", "    participant C as 고객", "    participant A as 에이전트",
         "    C->>A: (발화: " + intent + ")"]
    if p.get("handoff"):
        L += ["    A->>C: 상담사 연결 안내", "    A-->>상담사: 통화 전환"]
    else:
        for s in p["slots"]:
            L += ["    A->>C: " + s["question"], "    C->>A: (" + s["name"] + " 응답)"]
        if p["confirm"]:
            L += ["    A->>C: " + p["confirm_line"], "    C->>A: 네", "    A->>C: 처리 완료 안내"]
        else:
            L += ["    A->>C: 안내 응답"]
    return "\n".join(L)

scenarios = derive_acceptance_scenarios(EXAMPLE_CHARTER, DIALOG_PLAN)
assert len(scenarios) == sum(len(v) for v in EXAMPLE_CHARTER["golden_set"].values())   # 15발화
assert all(r["expected_first"] for r in scenarios)
assert all(r["expected_final"] in ("확인 턴 → 완료", "전화 전환", "응대 종료") for r in scenarios)

mmd = mermaid_sequence("reserve", DIALOG_PLAN)
assert "sequenceDiagram" in mmd and "어느 진료과" in mmd and "확정할까요" in mmd
assert "handoff" in mermaid_sequence("handoff", DIALOG_PLAN)

def derive_spike_template(charter):
    # 리스크마다 '2시간짜리 실험'의 골격 — 질문·방법·판정(숫자!)·기한은 팀이 채운다
    return [{"risk": r["risk"], "question": "", "method": "", "criterion": "", "due": ""}
            for r in charter["risks"]]

def validate_spikes(spikes, charter):
    assert len(spikes) >= len(charter["risks"]), "스파이크 위반 — 리스크보다 적다"
    for s in spikes:
        for k in ("question", "method", "criterion", "due"):
            assert s.get(k), "스파이크 위반 — 빈 칸: " + s["risk"][:16] + "…/" + k
        assert MEASURABLE.search(s["criterion"]), "스파이크 위반 — 판정 기준에 숫자 없음"
        assert DATE_RE.fullmatch(s["due"]), "스파이크 위반 — 기한 형식"
    return True

SPIKES = [
    {"risk": EXAMPLE_CHARTER["risks"][0]["risk"], "question": "고령 화자풍 발화에서 ASR이 쓸 만한가",
     "method": "골든셋 5발화를 gTTS slow=True로 합성 → ASR 라운드트립 CER 측정",
     "criterion": "CER 15% 이하", "due": "2026-08-07"},
    {"risk": EXAMPLE_CHARTER["risks"][1]["risk"], "question": "'다음 주 화요일' 류 상대 날짜를 규칙 파서가 감당하는가",
     "method": "날짜 표현 20건 수동 목록 → 정규화 함수 프로토타입 → 정답표 대조",
     "criterion": "정규화 정확도 90% 이상 (20건 중 18건)", "due": "2026-08-07"},
    {"risk": EXAMPLE_CHARTER["risks"][2]["risk"], "question": "교체 후보 LLM이 JSON 스키마를 지키는가",
     "method": "골든셋 15발화 → 후보 LLM → jsonschema 통과율 측정",
     "criterion": "스키마 통과율 95% 이상", "due": "2026-08-08"},
]
validate_spikes(SPIKES, EXAMPLE_CHARTER)
tpl = derive_spike_template(EXAMPLE_CHARTER)
assert len(tpl) == len(EXAMPLE_CHARTER["risks"])

def validate_milestones(ms, demo_day):
    dates = [m["due"] for m in ms]
    assert dates == sorted(dates) and all(d < demo_day for d in dates), "일정 위반 — 순서/시연일 역전"
    assert sum("pytest" in m["judge"] for m in ms) >= 2, "일정 위반 — pytest 판정 마일스톤 2개 미만"
    return True

MILESTONES = [
    {"name": "M1 스파이크 전건 판정", "due": "2026-08-08",
     "judge": "spikes.json 3건 결과 기록 + 실패 스파이크는 계약서 리스크 갱신"},
    {"name": "M2 핵심 기능(7-3)", "due": "2026-08-14",
     "judge": "pytest test_project.py → xfail 0, 전부 통과 (마커 제거 완료)"},
    {"name": "M3 파이프라인 통합(7-4)", "due": "2026-08-21",
     "judge": "pytest (전체 스위트+E2E) 종료 코드 0 + TTFA p95 기준 통과"},
    {"name": "M4 시연 리허설", "due": "2026-08-26",
     "judge": "acceptance_scenarios.json 전 시나리오 실연 녹화 1회 완주"},
]
validate_milestones(MILESTONES, "2026-08-28")
print("파생 산출물 검증 통과 ✅ — 시나리오 15 / mermaid / 스파이크 3(판정 숫자) / 마일스톤 4(pytest 2)")


In [ ]:
# ═══ 2.4 스텁 파생 + 계약 스켈레톤 — 7-3의 작업 지시서 (7-2) ✅ ═══
# ▶ 스텁: RealASR/LLM/TTS는 NotImplementedError — TODO가 7-3의 완료 정의다.
#   '손으로 다시 만들지 말 것': 7-2 파생 스텁이 구현 단계의 작업 지시서다.
def stub_source(wiring):
    # project_engines.py — 7-2 파생 스텁: 7-3의 작업 지시서 (손으로 다시 만들지 말 것)
    L = ['# project_engines.py — 7-2 파생 스텁: 7-3의 작업 지시서',
         'from aicc_bases import BaseASR, BaseLLM, BaseTTS', '']
    cls = {"asr": ("RealASR", "BaseASR", "transcribe", "self, audio, utt_id"),
           "llm": ("RealLLM", "BaseLLM", "generate", "self, prompt"),
           "tts": ("RealTTS", "BaseTTS", "synth", "self, text")}
    for kind, (cname, base, meth, sig) in cls.items():
        w = wiring[kind]
        L.append("class " + cname + "(" + base + "):")
        L.append('    name = "' + w["engine"] + '"')
        L.append("    def " + meth + "(" + sig + "):")
        for t in w["todo"]:
            L.append("        # TODO(7-3): " + t)
        L.append('        raise NotImplementedError("7-3에서 구현 — 위 TODO가 완료 정의다")')
        L.append("")
    return "\n".join(L)

wiring = {
    "asr": {"engine": "faster-whisper-small(int8)",
            "todo": ["골든셋 15발화 라운드트립 CER 측정", "reprompt 게이트 연동"]},
    "llm": {"engine": "EXAONE-4.0-1.2B(fp16)",
            "todo": ["INTENT_SCHEMA 파생 스키마 검증", "JSON 재시도 루프"]},
    "tts": {"engine": "Kokoro(82M)",
            "todo": ["16kHz mono float32 계약 준수", "상용구 캐시 연동"]},
}
stub = stub_source(wiring)
assert stub.count("class Real") == 3
assert stub.count("raise NotImplementedError") == 3
assert "faster-whisper-small(int8)" in stub and "TODO(7-3)" in stub

# ── 계약 스켈레톤: ASR/TTS 계약 (6-1 계승) — 7-3 어댑터가 통과해야 할 벽 ──
def make_asr_record(utt_id, text, language, confidence, engine, elapsed_ms):
    return {"utt_id": utt_id, "text": str(text), "language": str(language),
            "confidence": float(confidence), "engine": str(engine),
            "elapsed_ms": float(elapsed_ms)}

def validate_asr_contract(rec):
    assert rec["text"] and 0.0 <= rec["confidence"] <= 1.0
    return True

def make_tts_record(audio, sr, text):
    import numpy as np
    audio = np.asarray(audio, dtype=np.float32)
    return {"audio": audio, "sr": int(sr), "num_channels": 1,
            "dtype": str(audio.dtype), "text": text,
            "duration_s": round(len(audio) / sr, 3)}

def validate_tts_contract(rec):
    assert rec["sr"] == 16000 and rec["num_channels"] == 1 and rec["dtype"] == "float32"
    return True

import numpy as np
assert validate_asr_contract(make_asr_record("g_001", "예약하고 싶어요", "ko", 0.9, "e", 10.0))
assert validate_tts_contract(make_tts_record(np.zeros(16000, np.float32), 16000, "계약 검사용"))
print("스텁·계약 스켈레톤 검증 통과 ✅ — RealASR/LLM/TTS 3종 NotImplemented + 계약 2종")


## 2.5 📄 요약 — 카드 2장

### 카드 1: 프로젝트 진입 게이트 (7-1)
| 게이트 | 판정 | 기준 |
|---|---|---|
| Gate1 VRAM | 산수 | Σ×1.2+1.5 ≤ 16GB |
| Gate2 TTFA | 산수 | ASR+LLM+TTS ≤ 예산(1500ms) |
| Gate3 라이선스 | 정책 | 조합=max(랭크) · '미확인' 통과 불가 |
| Gate4 골든셋 | 계약 | 의도 5×발화 3 · snake_case · 8~80자 |

### 카드 2: 설계 파생 지도 (7-2)
```
Charter(SSOT) → schema · dialog_plan → mermaid + scenarios
              → spikes(숫자 판정) · milestones(pytest≥2) · engines 스텁(작업 지시서)
```

**교훈 4줄**:
1. 탈락은 **산수가 결정**한다 — 주장이 아니라 합계·추정·랭크로.
2. 범위는 "안 하는 것(non_goals)"이 지킨다 — 측정은 "숫자+단위"가 지킨다.
3. **정본은 하나**(Charter) — 사본은 손이 아니라 함수가 파생한다 (손 파생은 배신한다).
4. 완화책 없는 리스크는 걱정이지 관리가 아니다 — 스파이크는 '2시간 실험'으로 숫자 판정한다.


## 2.6 [REAL] 실물 실행 — 라운드트립(ASR→LLM→TTS) + 4대 게이트 실측

> 골든셋 발화 하나를 **실물 파이프라인**(mlx-whisper → Qwen3-4B → MeloTTS)으로 돌려
> CER·지연·TTFA를 실측하고, 2.0의 4대 게이트에 측정값을 반영합니다.
> 준비: `bash setup_apple_silicon.sh all`


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

if not ae.has("mlx_whisper", "mlx_lm") or not (ae.has("sherpa_onnx") or ae.has("melotts")):
    print("실물 스택 미설치 → 스킵.  bash setup_apple_silicon.sh asr llm tts (또는 all)")
else:
    import json, time
    import numpy as np

    def _cer(ref, hyp):
        r, h = ref.replace(" ", ""), hyp.replace(" ", "")
        dp = list(range(len(h) + 1))
        for i, rc in enumerate(r, 1):
            prev, dp[0] = dp[0], i
            for j, hc in enumerate(h, 1):
                cur = min(dp[j] + 1, dp[j - 1] + 1, prev + (rc != hc))
                prev, dp[j] = dp[j], cur
        return dp[len(h)] / len(r) if r else 0.0

    ref = "안녕하세요, 지난달 요금이 평소보다 많이 나온 것 같아서 확인 부탁드립니다."
    # 골든셋 발화 1건을 TTS로 합성 → 그 wav를 파이프라인 입력으로 (라운드트립)
    synth0 = ae.synth_korean(ref)
    assert synth0 is not None
    asr_wav = ae.save_wav(ae.HERE / "assets" / "_rt_asr.wav", synth0[0], synth0[1])

    # ① ASR — mlx-whisper
    import mlx_whisper
    t0 = time.perf_counter()
    asr_text = mlx_whisper.transcribe(str(asr_wav), path_or_hf_repo=ae.MODEL_CFG["asr_mlx"], language="ko")["text"]
    asr_ms = (time.perf_counter() - t0) * 1000
    print(f"① ASR  CER {_cer(ref, asr_text):.3f} | {asr_ms:.0f}ms | {asr_text[:36]}")

    # ② LLM — mlx-lm Qwen3-4B (INTENT 분류)
    from mlx_lm import load, generate
    model, tokenizer = load(ae.MODEL_CFG["llm_mlx"])
    msgs = [{"role": "system", "content": "다음 발화를 아래 JSON 스키마로만 응답하세요. 다른 텍스트 금지. "
             "스키마: {\"intent\": \"billing|tech_support|loss_suspend|plan_change|payment_change\", "
             "\"slots\": {}, \"reply\": \"고객 응대 문장\", \"handoff_to_human\": true|false}"},
            {"role": "user", "content": asr_text}]
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    t0 = time.perf_counter()
    raw = generate(model, tokenizer, prompt=prompt, max_tokens=256, verbose=False)
    llm_ms = (time.perf_counter() - t0) * 1000
    s = raw.strip().strip("```").removeprefix("json").strip()
    parsed = json.loads(s)
    reply = parsed["reply"]
    print(f"② LLM  intent={parsed['intent']} | {llm_ms:.0f}ms | {reply[:30]}")

    # ③ TTS — Supertonic(ko) / MeloTTS → 16k mono 로 규격화 후 계약
    t0 = time.perf_counter()
    synth = ae.synth_korean(reply)
    assert synth is not None, "TTS 미가용 (sherpa-onnx/melotts)"
    audio, sr = ae.to_16k_mono(synth[0], synth[1]), 16000
    tts_ms = (time.perf_counter() - t0) * 1000
    rec = make_tts_record(audio, sr, reply)
    assert validate_tts_contract(rec)
    print(f"③ TTS  {rec['duration_s']:.2f}s | {tts_ms:.0f}ms | 계약 통과")

    # ④ 4대 게이트에 실측 반영
    ttfa = asr_ms + llm_ms + tts_ms
    _orig = dict(MODEL_CATALOG)
    MODEL_CATALOG["mlx-whisper-large-v3-turbo(int4)"] = {"kind": "asr", "vram_gb": 1.6, "first_ms": round(asr_ms), "license": "상업 자유"}
    MODEL_CATALOG["Qwen3-4B(4bit)"] = {"kind": "llm", "vram_gb": 2.6, "first_ms": round(llm_ms), "license": "상업 자유"}
    MODEL_CATALOG["Supertonic-ko"] = {"kind": "tts", "vram_gb": 0.5, "first_ms": round(tts_ms), "license": "상업 자유"}
    stack = ["mlx-whisper-large-v3-turbo(int4)", "Qwen3-4B(4bit)", "Supertonic-ko"]
    print(f"\n④ 4대 게이트 — 실측 TTFA {ttfa:.0f}ms")
    vram_check(stack)
    ttfa_estimate(stack, "realtime_inbound")
    license_check(stack, "상업")
    MODEL_CATALOG.clear(); MODEL_CATALOG.update(_orig)
    print("실물 라운드트립 통과 ✅ — 골든셋이 실측으로 4대 게이트를 통과하는지 확인")


# 3. 실험 진행 방법 🧪

## 3-1. macOS에서 전부 실행 (이 노트 셀 순서)
```
2.0 4대 게이트 → 2.1 계약서·채점 → 2.2 스키마 파생·대화 계획
→ 2.3 시나리오·스파이크·마일스톤 → 2.4 스텁·계약 스켈레톤
```
별도 설치·키 없이 위에서 아래로 실행하면 됩니다.

## 3-2. 원본 실습 복원 (7-1 → 7-2)
1. **7-1**: 카탈로그 실측(`src` 기록) → 후보 주제 채점 → 4대 게이트 통과 → 계약서 작성(JSON 저장).
2. **7-2**: 계약서를 **복원**(`json.load`) → 재검증(게이트 다시 통과) → 스키마·대화계획·시나리오·스파이크·
   마일스톤·스텁 파생 → `pytest` 스위트 초록+xfail.
3. **7-3 이후**: 스텁의 `NotImplementedError`가 완료 정의 — TODO를 채우면 구현이다.

## 3-3. macOS 치환 요약
- 카탈로그 수치는 T4 기준 — **M4 Pro 48GB에서는 fp16으로 재측정** (Qwen3-8B는 48GB에서 상주 가능).
- `pytest` 스위트는 로컬에서 그대로 실행 가능 (`pip install pytest`).
- 모델 라운드트립 실측만 GPU/Colab 필요 — 게이트·파생 로직은 전부 macOS에서 검증.

## 3-4. 판단 기준 5종
1. **게이트 통과 여부** — 4대 게이트가 전부 통과하는가 (양방향 위반 케이스 포함).
2. **계약서 정합성** — 골든셋=의도 · 성공기준 측정가능 · non_goals 3+ · 리스크 완화책.
3. **SSOT 파생** — 스키마·시나리오가 손이 아닌 함수에서 나왔는가 (배신 테스트).
4. **스파이크 판정 가능성** — 각 스파이크에 숫자 판정 기준이 있는가.
5. **마일스톤 증명** — pytest 판정이 2개 이상 (사람 인상보다 종료코드).


# 4. 효율적 설계를 위한 아키텍처 🏛️

## 4-1. SSOT(단일 진실 공급원) 아키텍처

```
                ┌─────────────┐
                │  Charter    │  (7-1 계약서 = 정본)
                └──────┬──────┘
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
   INTENT_SCHEMA  dialog_plan     spikes/milestones
   (2.2 파생)     (2.2)           (2.3)
        │              │
        ▼              ▼
   project_engines  acceptance_scenarios
   스텁 (2.4)      + mermaid (2.3)
```

**원칙**: 사본은 손이 아니라 **함수**가 만든다. 정본이 바뀌면 파생물이 다시 생성된다.

## 4-2. 게이트가 먼저, 계약서가 그 다음
- 주제는 **게이트(산수·정책)**를 통과한 뒤에야 계약서가 된다 — 반대 순서는 문서만 많아진다.
- 계약서는 서명한 날이 아니라 **검사하는 날마다** 유효 — `verification_log`의 재검증 날짜가 증거.

## 4-3. macOS 적용 아키텍처
- 계약·파생·게이트 로직은 전부 macOS 실행 — 모델 실측만 GPU.
- 스텁이 "7-3 작업 지시서" 역할 — 구현 단계에서도 계약(어댑터 벽)은 그대로 재사용.

## 4-4. 최종 판정
- **7-1**: 4대 게이트 + 계약서가 "완성될 수 있는" 프로젝트를 고른다.
- **7-2**: 설계 산출물은 전부 정본에서 파생 — 손 파생의 배신을 함수로 차단.
- **7-3~7-5**: 스텁·시나리오·마일스톤이 구현·통합·시연의 완료 정의가 된다.
